In [3]:
import cv2
import numpy as np

In [50]:
def select_line_event(event, x, y, flags, param):
    point_num = len(param[1])
    imgs = param[0]
    
    if event == cv2.EVENT_LBUTTONDOWN:

        if point_num < 2:
            param[1].append((x,y))
            param[2] -= 1
            
            if point_num == 0:
                cv2.circle(imgs[1], (x, y), 2, (255, 255, 255), -1)
                param[2] = imgs[1]
                cv2.imshow("image", param[2])
                
            elif point_num == 1:
                cv2.line(imgs[2], param[1][0], (x, y), (255, 255, 255), 1)
                
                cv2.circle(imgs[2], (x, y), 2, (255, 255, 255), -1)
                param[2] = imgs[2]
                cv2.imshow("image", param[2])

    elif event == cv2.EVENT_RBUTTONDOWN:
        if point_num > 0:
            (x,y) = param[1].pop()
        
            cv2.circle(img, (x, y), 2, (0, 0, 0), -1)
            cv2.imshow("image", img)
    

def select_line(img_path):
    img = cv2.imread(img_path)
    if img is None:
        print("Error: Image not found. Please check the file path.")
        exit()

    #img_copy = img.copy()
    #img_copy_1 = img.copy()
    #img_copy_2 =  img.copy()
    img_copy = [img.copy(), img.copy(), img.copy()]
    cur_img = img_copy[0]
    i = 0

    
    points = []

    cv2.namedWindow("image")
    cv2.setMouseCallback("image", select_line_event, [img_copy, points, cur_img])

    while True:       
        cv2.imshow("image", cur_img)

        if cv2.waitKey(1) & 0xFF == 8:
            if i > 0:
                i = i-1

        if cv2.waitKey(1) & 0xFF in [27,13]:
            break
            
    cv2.destroyAllWindows()
    print(points)

In [51]:
select_line('./data/2026.03.03/run-3/38.jpg')

[(213, 484), (246, 482)]


In [2]:
# Global variables to store points
drawing = False # true if mouse is pressed
points = []

def select_poly_roi(event, x, y, flags, param):
    global points, drawing, img_copy

    if event == cv2.EVENT_LBUTTONDOWN:
        drawing = True
        points.append((x, y))
        # Draw a small circle to mark the point
        cv2.circle(img_copy, (x, y), 3, (0, 255, 0), -1)
        cv2.imshow("image", img_copy)

    elif event == cv2.EVENT_MOUSEMOVE:
        if drawing:
            # Optionally draw a continuous line as a guide during selection
            # A cleaner approach is to just show the points and connect on finish
            pass

    elif event == cv2.EVENT_LBUTTONUP:
        drawing = False
        # Connect the last point to the first to close the polyline
        # This is handled later when creating the mask

# Load image
img = cv2.imread('./data/2026.03.03/run-3/38.jpg') # Replace with your image file
if img is None:
    print("Error: Image not found. Please check the file path.")
    exit()

img_copy = img.copy()
cv2.namedWindow("image")
cv2.setMouseCallback("image", select_poly_roi)

print("Click to define polygon vertices. Press 'c' to crop the ROI or 'ESC' to exit.")

while True:
    cv2.imshow("image", img_copy)
    k = cv2.waitKey(1) & 0xFF

    if k == 27: # ESC key to exit
        break
    elif k == ord('c'): # 'c' key to confirm and crop
        if len(points) >= 3:
            # Create a mask
            mask = np.zeros(img.shape[:2], dtype=np.uint8)
            # Reshape points for fillPoly
            pts = np.array(points, dtype=np.int32)
            pts = pts.reshape((-1, 1, 2))
            # Fill the polygon area with white (255)
            cv2.fillPoly(mask, [pts], 255)

            # Apply the mask to the original image
            # The result will only show the region within the polygon
            roi = cv2.bitwise_and(img, img, mask=mask)
            
            # Find the bounding box of the ROI to crop the image
            x, y, w, h = cv2.boundingRect(pts)
            cropped_roi = roi[y:y+h, x:x+w]

            cv2.imshow("Cropped ROI", cropped_roi)
            cv2.waitKey(0) # Wait for any key press to close cropped ROI window
        else:
            print("Please select at least 3 points for a polygon.")

cv2.destroyAllWindows()

Click to define polygon vertices. Press 'c' to crop the ROI or 'ESC' to exit.
